# Gator Convergent language scaling lab

This notebook runs controlled A/B/C language experiments with the connected Convergent architecture. Edit the candidate table to vary width, shared-node count, persistent lane count, graph degree/layout/rewiring, and relay depth independently. The preview cell calculates exact trainable parameters before any model is launched.

The supplied candidates hold total size near 130k while testing more/fewer lanes and more skinny/fewer fat nodes. Relay count changes recurrent compute without changing trainable parameters. Use the same corpus, seed, batches, optimizer horizon, and evaluation set for every candidate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BUNDLE = '/content/drive/MyDrive/GTR/gator_convergent_colab_bundle.zip'
WORKDIR = '/content/GTR_Convergent'
!rm -rf "$WORKDIR"
!mkdir -p "$WORKDIR"
!unzip -q -o "$BUNDLE" -d "$WORKDIR"
%cd $WORKDIR

import torch
print('torch', torch.__version__)
print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')


In [ ]:
# Shared experimental controls. Keep these fixed during a topology/shape A/B.
STEPS = 300000
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LOG_EVERY = 1000
LEARNING_RATE = 0.003
SCHEDULER_HORIZON = 300000
SEED = 0
COMPILE = True

DATA = '/content/drive/MyDrive/GTR/training_data/smoltalk_everyday.jsonl'
LMSYS = '/content/drive/MyDrive/GTR/training_data/lmsys_chat_1m.jsonl'
TULU3 = '/content/drive/MyDrive/Sonic/training_data/chat/tulu3_sft.jsonl'
PREFERENCES = '/content/drive/MyDrive/Sonic/training_data/chat/smoltalk_preference.jsonl'
CHECKPOINT_ROOT = '/content/drive/MyDrive/GTR/experimental_checkpoints/convergent_scaling'

# Select exactly one clean sweep. Every row exposes all architecture knobs.
# `formation` is the next controlled run. Its global D30 baseline already exists.
EXPERIMENT = 'routing'  # 'routing', 'formation', 'capacity', 'shape', 'compute_width', 'lane_density', 'voters', 'connections', or 'relays'

EXPERIMENTS = {
    # Clean routing A/B: legacy oracle versus factorized global ingress and active execution.
    'routing': [
        dict(name='routing_legacy_d20_n66_l10_r2', width=20, nodes=66, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.50,
             embedding=32, vote_mode='slot', router_mode='private_full',
             ingress_mode='fixed'),
        dict(name='routing_shared_r16_dense_d20_n66_l10_r2', width=20, nodes=66, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.50,
             embedding=32, vote_mode='slot', router_mode='shared_low_rank',
             router_rank=16, ingress_mode='learned'),
        dict(name='routing_shared_r16_k4_d20_n66_l10_r2', width=20, nodes=66, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.50,
             embedding=32, vote_mode='slot', router_mode='shared_low_rank',
             router_rank=16, ingress_mode='learned', active_top_k=4,
             active_mass_threshold=0.01),
    ],
    # Approximately fixed 130k total: lane count and node width/population.
    'shape': [
        dict(name='baseline_d20_n66_l10_r2', width=20, nodes=66, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.50),
        dict(name='more_lanes_d20_n49_l14_r2', width=20, nodes=49, lanes=14,
             degree=3, relays=2, layout='small_world', rewire=0.50),
        dict(name='fewer_lanes_d20_n81_l6_r2', width=20, nodes=81, lanes=6,
             degree=3, relays=2, layout='small_world', rewire=0.50),
        dict(name='skinny_nodes_d16_n109_l10_r2', width=16, nodes=109, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.50),
        dict(name='fat_nodes_d24_n40_l10_r2', width=24, nodes=40, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.50),
    ],
    # Holds topology, lanes, lexical width, and voting fixed; only payload/node width changes.
    'compute_width': [
        dict(name='width_d20_n66_l10_r2', width=20, nodes=66, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.50,
             embedding=32, vote_mode='global'),
        dict(name='width_d30_n66_l10_r2', width=30, nodes=66, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.50,
             embedding=32, vote_mode='global'),
    ],
    # Existing D30/global seed-zero run is the exact baseline. Run only the
    # 300-parameter slot controller so Blackwell time is not spent repeating it.
    'formation': [
        dict(name='formation_slot_d30_n66_l10_r2', width=30, nodes=66, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.50,
             embedding=32, vote_mode='slot'),
    ],
    # Under-compute test for 32 simultaneous response slots. D50 is exactly
    # 350,342 parameters; D54 (391,390) remains the follow-up, not a co-run.
    'capacity': [
        dict(name='capacity_d50_n66_l10_r2', width=50, nodes=66, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.50,
             embedding=32, vote_mode='global'),
    ],
    # Tests whether lane count requires proportional shared-graph territory.
    'lane_density': [
        dict(name='density_l10_n66_d20_r2', width=20, nodes=66, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.50),
        dict(name='density_l20_n66_d20_r2', width=20, nodes=66, lanes=20,
             degree=3, relays=2, layout='small_world', rewire=0.50),
        dict(name='density_l14_n90_d20_r2', width=20, nodes=90, lanes=14,
             degree=3, relays=2, layout='small_world', rewire=0.50),
        dict(name='density_l20_n80_d20_r2', width=20, nodes=80, lanes=20,
             degree=3, relays=2, layout='small_world', rewire=0.50),
        dict(name='density_l20_n132_d20_r2', width=20, nodes=132, lanes=20,
             degree=3, relays=2, layout='small_world', rewire=0.50),
    ],
    # Exact terminal-jury A/B at the same graph and lexical width.
    'voters': [
        dict(name='voter_global_l16_n90_d20_r2', width=20, nodes=90, lanes=16,
             degree=3, relays=2, layout='small_world', rewire=0.50,
             embedding=32, vote_mode='global'),
        dict(name='voter_prompt_l16_n90_d20_r2', width=20, nodes=90, lanes=16,
             degree=3, relays=2, layout='small_world', rewire=0.50,
             embedding=32, vote_mode='prompt'),
        dict(name='voter_slot_l16_n90_d20_r2', width=20, nodes=90, lanes=16,
             degree=3, relays=2, layout='small_world', rewire=0.50,
             embedding=32, vote_mode='slot'),
    ],
    # Degree rows compensate node count to remain near 130k. Rewire rows are exact A/Bs.
    'connections': [
        dict(name='degree2_d20_n72_r2', width=20, nodes=72, lanes=10,
             degree=2, relays=2, layout='small_world', rewire=0.50),
        dict(name='degree3_sw25_d20_n66_r2', width=20, nodes=66, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.25),
        dict(name='degree3_sw50_d20_n66_r2', width=20, nodes=66, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.50),
        dict(name='degree3_sw100_d20_n66_r2', width=20, nodes=66, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=1.00),
        dict(name='degree5_d20_n56_r2', width=20, nodes=56, lanes=10,
             degree=5, relays=2, layout='small_world', rewire=0.50),
    ],
    # Relay depth changes recurrent compute, not trainable parameter count.
    'relays': [
        dict(name='baseline_d20_n66_l10_r1', width=20, nodes=66, lanes=10,
             degree=3, relays=1, layout='small_world', rewire=0.50),
        dict(name='baseline_d20_n66_l10_r2', width=20, nodes=66, lanes=10,
             degree=3, relays=2, layout='small_world', rewire=0.50),
        dict(name='baseline_d20_n66_l10_r4', width=20, nodes=66, lanes=10,
             degree=3, relays=4, layout='small_world', rewire=0.50),
        dict(name='baseline_d20_n66_l10_r8', width=20, nodes=66, lanes=10,
             degree=3, relays=8, layout='small_world', rewire=0.50),
    ],
}
CANDIDATES = EXPERIMENTS[EXPERIMENT]


In [ ]:
from gator_convergent import ConvergentGator

def parameter_count(candidate):
    return ConvergentGator.shared_language_parameter_count_for(
        vocab_size=2048,
        embedding_dim=candidate.get('embedding', 32),
        n_nodes=candidate['nodes'],
        d_model=candidate['width'],
        lanes=candidate['lanes'],
        degree=candidate['degree'],
        router_mode=candidate.get('router_mode', 'private_full'),
        router_rank=candidate.get('router_rank', 16),
        ingress_mode=candidate.get('ingress_mode', 'fixed'),
        terminal_vote_mode=candidate.get('vote_mode', 'global'),
    )

for candidate in CANDIDATES:
    candidate['parameters'] = parameter_count(candidate)
    print(
        f"{candidate['name']}: params={candidate['parameters']:,} "
        f"D={candidate['width']} N={candidate['nodes']} "
        f"lanes={candidate['lanes']} degree={candidate['degree']} "
        f"R={candidate['relays']} E={candidate.get('embedding', 32)} "
        f"vote={candidate.get('vote_mode', 'global')} {candidate['layout']} "
        f"rewire={candidate['rewire']:.2f} "
        f"router={candidate.get('router_mode', 'private_full')} "
        f"ingress={candidate.get('ingress_mode', 'fixed')} "
        f"active_k={candidate.get('active_top_k')}"
    )


In [ ]:
import pathlib
import subprocess
import sys

pathlib.Path(CHECKPOINT_ROOT).mkdir(parents=True, exist_ok=True)

for candidate in CANDIDATES:
    run_name = f"{candidate['name']}_seed{SEED}_{STEPS}steps"
    checkpoint_dir = f"{CHECKPOINT_ROOT}/{run_name}"
    command = [
        sys.executable, '-u', 'gator_language_train.py',
        '--data', DATA,
        '--lmsys-data', LMSYS,
        '--streaming-weight', '1.0',
        '--streaming-exhaustion', 'restart',
        '--primary-stream-weight', '1',
        '--primary-shuffle-buffer', '8192',
        '--lmsys-source-weight', '4',
        '--chat-data', TULU3, '--chat-source-weight', '3',
        '--chat-data', PREFERENCES, '--chat-source-weight', '2',
        '--lmsys-load-mode', 'stream',
        '--lmsys-shuffle-buffer', '16384',
        '--chat-shuffle-buffer', '8192',
        '--lmsys-validation-conversations', '512',
        '--lmsys-max-characters', '512',
        '--chat-max-characters', '512',
        '--lmsys-min-ascii-letter-ratio', '0.9',
        '--lmsys-min-response-characters', '8',
        '--lmsys-require-user-prompt',
        '--require-complete-context',
        '--trim-context-turns',
        '--deduplicate-examples',
        '--architecture', 'convergent',
        '--steps', str(STEPS),
        '--batch-size', str(BATCH_SIZE),
        '--eval-batch-size', str(EVAL_BATCH_SIZE),
        '--log-every', str(LOG_EVERY),
        '--logging-slim',
        '--vocab-size', '2048',
        '--context-tokens', '64',
        '--response-tokens', '32',
        '--embedding-dim', str(candidate.get('embedding', 32)),
        '--d-model', str(candidate['width']),
        '--nodes', str(candidate['nodes']),
        '--graph-degree', str(candidate['degree']),
        '--lanes', str(candidate['lanes']),
        '--terminal-vote-mode', candidate.get('vote_mode', 'global'),
        '--router-mode', candidate.get('router_mode', 'private_full'),
        '--router-rank', str(candidate.get('router_rank', 16)),
        '--ingress-mode', candidate.get('ingress_mode', 'fixed'),
        '--active-node-mass-threshold', str(candidate.get('active_mass_threshold', 0.0)),
        '--relays', str(candidate['relays']),
        '--connection-layout', candidate['layout'],
        '--connection-seed', str(SEED),
        '--connection-rewire', str(candidate['rewire']),
        '--skip-first-assistant',
        '--scheduler', 'cosine',
        '--scheduler-total-steps', str(SCHEDULER_HORIZON),
        '--minimum-lr', '0.0001',
        '--learning-rate', str(LEARNING_RATE),
        '--seed', str(SEED),
        '--batch-pipeline', 'prefetch',
        '--prefetch-batches', '20',
        '--float32-matmul-precision', 'high',
        '--cudnn-tf32',
        '--checkpoint-dir', checkpoint_dir,
    ]
    if candidate.get('active_top_k') is not None:
        command.extend(['--active-node-top-k', str(candidate['active_top_k'])])
    if COMPILE:
        command.append('--compile')
    print('\n' + '=' * 100, flush=True)
    print(f"START {run_name} parameters={candidate['parameters']:,}", flush=True)
    print(' '.join(command), flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    print(f"DONE {run_name}", flush=True)


## Clean one-lever follow-ups

- **Lanes:** compare the baseline, 14-lane, and 6-lane rows. Node counts compensate for lane parameters.
- **Width versus node population:** compare baseline, D16/N109, and D24/N40.
- **Relays:** duplicate one row and change only `relays` to 1, 4, or 8. Shared recurrent weights keep parameter count fixed.
- **Connections:** duplicate one row and change only `degree`, `layout`, or `rewire`. Degree changes parameter count, so use the preview before interpreting the result.
- **Routing:** run all three `routing` rows in order: legacy control, shared router plus learned ingress without pruning, then the same factorized router with top-4/0.01 active mass.
- **Seeds:** rerun the winning shape with `SEED = 1` and `SEED = 2` before declaring a winner.

For a fair short screen, rank candidates by best validation PPL, content/context quality in `formed`, and wall time to a common threshold—not final PPL alone and not steps/second alone.

In [ ]:
# Compact best-checkpoint comparison after the selected sweep finishes.
import pandas as pd
import torch

rows = []
for candidate in CANDIDATES:
    run_name = f"{candidate['name']}_seed{SEED}_{STEPS}steps"
    checkpoint = f"{CHECKPOINT_ROOT}/{run_name}/best.pt"
    state = torch.load(checkpoint, map_location='cpu', weights_only=False)
    validation = state['validation']
    rows.append({
        'name': candidate['name'],
        'params': candidate['parameters'],
        'best_step': state['step'],
        'ppl': validation['perplexity'],
        'loss': validation['loss'],
        'accuracy': validation['accuracy'],
        'content_ppl': validation['content_perplexity'],
        'content_accuracy': validation['content_accuracy'],
        'coverage': validation['formation_coverage'],
        'aligned_content_f1': validation['aligned_content_f1'],
        'formed': validation['prediction'],
    })

results = pd.DataFrame(rows).sort_values(['ppl', 'content_ppl'])
print('\nFINAL SWEEP RESULTS')
print(results.to_string(index=False))
display(results)
results.to_csv(f"{CHECKPOINT_ROOT}/{EXPERIMENT}_seed{SEED}_{STEPS}steps.csv", index=False)

print('\nFIXED SEMANTIC PROBES')
for candidate in CANDIDATES:
    run_name = f"{candidate['name']}_seed{SEED}_{STEPS}steps"
    checkpoint = f"{CHECKPOINT_ROOT}/{run_name}/best.pt"
    print('\n' + candidate['name'])
    subprocess.run([sys.executable, 'language_probe.py', checkpoint], check=True)
